In [5]:
import sys
import importlib
sys.path.append('../')  # Adjust the path as needed

import utilities.functions as functions
import utilities.plot as plot

# Reload the module to reflect the changes
importlib.reload(functions)
importlib.reload(plot)

<module 'utilities.plot' from '/Users/xuechenkan/potts_model_test/ms1/../utilities/plot.py'>

In [6]:
IN_seq_path = 'IN/data/in.reduce4.seq'
PR_seq_path = 'PR/data/pr.exper.reduce4.seq'    
RT_seq_path = 'RT/data/rt.reduce4.seq'

IN_all_seq = functions.read_seq('IN/data/in.reduce4.seq')
PR_all_seq = functions.read_seq('PR/data/pr.exper.reduce4.seq')
RT_all_seq = functions.read_seq('RT/data/rt.reduce4.seq')

IN_consensus = 'IN/data/in.consensus.reduce4.seq'
with open(IN_consensus, 'r') as f:
    IN_consensus_seq = f.read().strip()
# print("IN consensus sequence:", IN_consensus_seq)
PR_consensus = 'PR/data/pr.consensus.reduce4.seq'
with open(PR_consensus, 'r') as f:
    PR_consensus_seq = f.read().strip()
RT_consensus = 'RT/data/rt.consensus.reduce4.seq'
with open(RT_consensus, 'r') as f:
    RT_consensus_seq = f.read().strip()

# print(IN_consensus_seq)
    
IN_redux = functions.get_redu_dict('IN/data/in.reduce4.redux',1)
PR_redux = functions.get_redu_dict('PR/data/pr.reduce4.redux',0)
RT_redux = functions.get_redu_dict('RT/data/rt.reduce4.redux',0)

IN_J = functions.load_J_dict('IN/data/J.npy',1,263)
PR_J = functions.load_J_dict('PR/data/J_PR.npy',1,99)
RT_J = functions.load_J_dict('RT/data/J_RT.npy',39,226)

IN_all_seq_unreduced = functions.read_seq('IN/data/in.fullseq')
PR_all_seq_unreduced = functions.read_seq('PR/data/pr.exper.fullseq')
RT_all_seq_unreduced = functions.read_seq('RT/data/rt.fullseq')

In [ ]:
import csv

def output_probs(prefix, min_pos, max_pos, all_seq, consensus_seq, redux, pairs, weights_path, J, output_csv):
    """
    Process epistasis data for a given prefix (e.g., IN, PR, RT).

    Parameters:
        prefix (str): Prefix for the dataset (e.g., 'IN', 'PR', 'RT').
        all_seq (list): List of all sequences.
        consensus_seq (str): Consensus sequence.
        redux (dict): Reduction dictionary.
        pairs (list): List of mutation pairs.
        weights_path (str): Path to the weights file.
        J (dict): Interaction matrix.
        output_csv (str): Output CSV file name.
    """
    len_all_seqs = len(all_seq)

    # Read weights from the file
    with open(weights_path, 'r') as f:
        weights = [float(line.strip()) for line in f]

    # Ensure the weights list matches the all_seq list
    assert len(weights) == len(all_seq), "Weights and sequences must have the same length."

    alphabet = ['A','B','C','D']
    csv_total_data = []
    csv_data = []
    csv_weighted_data = []

    for ipair,pair in enumerate(pairs):
        print(f"Processing pair {ipair+1}/{len(pairs)}: {pair}")
        pair1, pair2 = functions.split_pairs(pair)
        p1_reduced = functions.unreduced_to_reduced(redux, pair1)
        p2_reduced = functions.unreduced_to_reduced(redux, pair2)
        wt1, pos1, mt1 = functions.split_pair(p1_reduced)
        wt2, pos2, mt2 = functions.split_pair(p2_reduced)

        p1_alternative = []
        p2_alternative = []

        for alt_aa in alphabet:
            if alt_aa == mt1 or alt_aa == wt1:
                continue
            p1_alternative.append(wt1 + str(pos1) + alt_aa)
        for alt_aa in alphabet:
            if alt_aa == mt2 or alt_aa == wt2:
                continue
            p2_alternative.append(wt2 + str(pos2) + alt_aa)
        ########
        flip_counts = 0
        flip_without_DMC_count = 0
        flip_with_DMC_count = 0
        flip_with_one_mut_count = 0
        
        ########
        non_flip_counts = 0
        non_flip_without_DMC_count = 0
        non_flip_with_DMC_count = 0
        non_flip_with_one_mut_count = 0

        ########
        gof_counts = 0
        gof_without_DMC_count = 0
        gof_with_DMC_count = 0
        gof_with_one_mut_count = 0

        ########
        rescue_count = 0
        rescue_without_DMC_count = 0
        rescue_with_DMC_count = 0
        rescue_with_one_mut_count = 0

        ########
        compensatory_counts = 0
        compensatory_without_DMC_count = 0
        compensatory_with_DMC_count = 0
        compensatory_with_one_mut_count = 0

        ########
        noncompensatory_counts = 0
        noncompensatory_without_DMC_count = 0
        noncompensatory_with_DMC_count = 0
        noncompensatory_with_one_mut_count = 0

        flip_seqs = []
        flip_weights = []

        non_flip_seqs = []
        non_flip_weights = []

        gof_seqs = []
        gof_weights = []

        rescue_seqs = []
        rescue_weights = []

        comp_seqs = []
        comp_weights = []

        noncomp_seqs = []
        non_comp_weights = []
    ##############
        consensus_result = functions.calculate_dde_v2(p1_reduced, p2_reduced, consensus_seq, J, min_pos, max_pos)
        consensus_pair1_de, consensus_pair2_de, consensus_pair12_de, consensus_pair12_dde = consensus_result
        consensus_de_diff = consensus_pair1_de - consensus_pair2_de
    ##############
        total_counts = 0
        total_with_DMC_count = 0
        all_seqs = []

        curr_i = 0
        for curr_i, seq in enumerate(all_seq):
            # print(f"Processing sequence {curr_i+1}/{len(all_seq)}")
            result_merged = functions.calculate_dde_v2(p1_reduced, p2_reduced, seq, J, min_pos, max_pos)

            if result_merged is None:
                continue
            pair1_de, pair2_de, pair12_de, pair12_dde = result_merged

            total_counts += 1
            all_seqs.append(seq)
            if seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2:
                total_with_DMC_count += 1
            result_merged = functions.calculate_dde_v2(p1_reduced, p2_reduced, seq, J, min_pos, max_pos)

            de12_alts_mut1 = []
            de12_alts_mut2 = []
            for alt_mutants in p1_alternative:
                de12_alts = functions.calculate_delta_e_double(alt_mutants,p2_reduced,seq,J,min_pos,max_pos)
                de12_alts_mut1.append(de12_alts)
            for alt_mutants in p2_alternative:
                de12_alts = functions.calculate_delta_e_double(p1_reduced,alt_mutants,seq,J,min_pos,max_pos)
                de12_alts_mut2.append(de12_alts)
            
            max_alt_de12 = max(max(de12_alts_mut1), max(de12_alts_mut2))
    ###############
            if result_merged is None:
                continue
            pair1_de, pair2_de, pair12_de, pair12_dde = result_merged
            de_diff_seq = pair1_de - pair2_de
    ###############
    #flip/nonflip block TODO there are logical errors for one mutation count
            if consensus_de_diff * de_diff_seq < 0: #FLIPPED!!
                flip_counts += 1
                flip_seqs.append(seq)
                flip_weights.append(weights[curr_i])
                
                if seq[pos1 - min_pos] == wt1 and seq[pos2 - min_pos] == wt2:
                    flip_without_DMC_count += 1
                elif seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2:
                    flip_with_DMC_count += 1
                else:
                    flip_with_one_mut_count += 1

            elif consensus_de_diff * de_diff_seq > 0: #NON-FLIPPED
                non_flip_counts += 1
                non_flip_seqs.append(seq)
                non_flip_weights.append(weights[curr_i])

                if seq[pos1 - min_pos] == wt1 and seq[pos2 - min_pos] == wt2:
                    non_flip_without_DMC_count += 1
                elif seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2:
                    non_flip_with_DMC_count += 1
                else:
                    non_flip_with_one_mut_count += 1
    ###############
            if pair1_de < pair12_de and pair2_de < pair12_de and pair12_de > 0 and pair12_de > max_alt_de12:
                gof_counts += 1
                gof_seqs.append(seq)
                gof_weights.append(weights[curr_i])

                if seq[pos1 - min_pos] == wt1 and seq[pos2 - min_pos] == wt2:
                    gof_without_DMC_count += 1
                elif seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2:
                    gof_with_DMC_count += 1
                else:
                    gof_with_one_mut_count += 1

            elif pair1_de < pair12_de and pair2_de < pair12_de:
                rescue_count += 1
                rescue_seqs.append(seq)
                rescue_weights.append(weights[curr_i])

                if seq[pos1 - min_pos] == wt1 and seq[pos2 - min_pos] == wt2:
                    rescue_without_DMC_count += 1
                elif seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2:
                    rescue_with_DMC_count += 1
                else:
                    rescue_with_one_mut_count += 1

            elif pair1_de < pair12_de or pair2_de < pair12_de:
                compensatory_counts += 1
                comp_seqs.append(seq)
                comp_weights.append(weights[curr_i])

                if seq[pos1 - min_pos] == wt1 and seq[pos2 - min_pos] == wt2:
                    compensatory_without_DMC_count += 1
                elif seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2:
                    compensatory_with_DMC_count += 1
                else:
                    compensatory_with_one_mut_count += 1
            else:
                noncompensatory_counts += 1
                noncomp_seqs.append(seq)
                non_comp_weights.append(weights[curr_i])

                if seq[pos1 - min_pos] == wt1 and seq[pos2 - min_pos] == wt2:
                    noncompensatory_without_DMC_count += 1
                elif seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2:
                    noncompensatory_with_DMC_count += 1
                else:
                    noncompensatory_with_one_mut_count += 1
            curr_i += 1
        p_SH_all = [functions.calculate_double_mutant_probablity_v2(seq, p1_reduced, p2_reduced, J, min_pos, max_pos) for seq in all_seqs]
        average_p_all = sum(p_SH_all) / len_all_seqs if p_SH_all else 0
        weighted_all_prob = sum(p * w for p, w in zip(p_SH_all, weights[:len_all_seqs])) / sum(weights[:len_all_seqs]) if p_SH_all else 0
        
        #total weights sum and weights sum for sequences with DMC
        weights_sum = sum(weights[:len_all_seqs])
        weights_with_DMC_sum = sum((1 if seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2 else 0) * w for seq, w in zip(all_seqs, weights[:len(all_seqs)]))

        # Calculate the actual_p values
        actual_p_all = total_with_DMC_count / total_counts if total_counts > 0 else 0
        actual_p_all_weighted = weights_with_DMC_sum / weights_sum if all_seqs else 0

        # Append data
        csv_total_data.append([pair,'Total', total_counts,total_with_DMC_count, average_p_all, actual_p_all, average_p_all, actual_p_all,weights_sum,weights_with_DMC_sum, weighted_all_prob, actual_p_all_weighted,weighted_all_prob, actual_p_all_weighted])
        # Calculate average probabilities
        p_SH_flip = [functions.calculate_double_mutant_probablity_v2(seq, p1_reduced, p2_reduced, J, min_pos, max_pos) for seq in flip_seqs]
        average_p_flip_subcategory = sum(p_SH_flip) / len(p_SH_flip) if p_SH_flip else 0
        average_p_flip_total = sum(p_SH_flip) / len_all_seqs if len_all_seqs else 0
        weighted_flip_prob_subcategory = sum(p * w for p, w in zip(p_SH_flip, flip_weights)) / sum(flip_weights) if flip_weights else 0
        weighted_flip_prob_total = sum(p * w for p, w in zip(p_SH_flip, flip_weights)) / sum(weights) if flip_weights else 0

        p_SH_nonflip = [functions.calculate_double_mutant_probablity_v2(seq, p1_reduced, p2_reduced, J, min_pos, max_pos) for seq in non_flip_seqs]
        average_p_nonflip_subcategory = sum(p_SH_nonflip) / len(p_SH_nonflip) if p_SH_nonflip else 0
        average_p_nonflip_total = sum(p_SH_nonflip) / len_all_seqs if len_all_seqs else 0
        weighted_nonflip_prob_subcategory = sum(p * w for p, w in zip(p_SH_nonflip, non_flip_weights)) / sum(non_flip_weights) if non_flip_weights else 0
        weighted_nonflip_prob_total = sum(p * w for p, w in zip(p_SH_nonflip, non_flip_weights)) / sum(weights) if non_flip_weights else 0 

        p_SH_gofs = [functions.calculate_double_mutant_probablity_v2(seq, p1_reduced, p2_reduced, J, min_pos, max_pos) for seq in gof_seqs]
        average_p_gof_subcategory = sum(p_SH_gofs) / len(p_SH_gofs) if p_SH_gofs else 0
        average_p_gof_total = sum(p_SH_gofs) / len_all_seqs if len_all_seqs else 0
        weighted_gof_prob_subcategory = sum(p * w for p, w in zip(p_SH_gofs, gof_weights)) / sum(gof_weights) if gof_weights else 0
        weighted_gof_prob_total = sum(p * w for p, w in zip(p_SH_gofs, gof_weights)) / sum(weights) if gof_weights else 0
        
        p_SH_rescues = [functions.calculate_double_mutant_probablity_v2(seq, p1_reduced, p2_reduced, J, min_pos, max_pos) for seq in rescue_seqs]
        average_p_rescue_subcategory = sum(p_SH_rescues) / len(p_SH_rescues) if p_SH_rescues else 0
        average_p_rescue_total = sum(p_SH_rescues) / len_all_seqs if len_all_seqs else 0
        weighted_rescue_prob_subcategory = sum(p * w for p, w in zip(p_SH_rescues, rescue_weights)) / sum(rescue_weights) if rescue_weights else 0
        weighted_rescue_prob_total = sum(p * w for p, w in zip(p_SH_rescues, rescue_weights)) / sum(weights) if rescue_weights else 0

        # average_p_gof_rescue = (sum(p_SH_gofs) + sum(p_SH_rescues)) / (len(p_SH_gofs) + len(p_SH_rescues)) if (len(p_SH_gofs) + len(p_SH_rescues)) > 0 else 0
        p_SH_comps = [functions.calculate_double_mutant_probablity_v2(seq, p1_reduced, p2_reduced, J, min_pos, max_pos) for seq in comp_seqs]
        average_p_compensatory_subcategory = sum(p_SH_comps) / len(p_SH_comps) if p_SH_comps else 0
        average_p_compensatory_total = sum(p_SH_comps) / len_all_seqs if len_all_seqs else 0
        weighted_comp_prob_subcategory = sum(p * w for p, w in zip(p_SH_comps, comp_weights)) / sum(comp_weights) if comp_weights else 0
        weighted_comp_prob_total = sum(p * w for p, w in zip(p_SH_comps, comp_weights)) / sum(weights)

        p_SH_noncomps = [functions.calculate_double_mutant_probablity_v2(seq, p1_reduced, p2_reduced, J, min_pos, max_pos) for seq in noncomp_seqs]
        average_p_non_comp_subcategory = sum(p_SH_noncomps) / len(p_SH_noncomps) if p_SH_noncomps else 0
        average_p_noncomp_total = sum(p_SH_noncomps) / len_all_seqs if len_all_seqs else 0
        weighted_noncomp_prob_subcategory = sum(p * w for p, w in zip(p_SH_noncomps, non_comp_weights)) / sum(non_comp_weights) if non_comp_weights else 0
        weighted_noncomp_prob_total = sum(p * w for p, w in zip(p_SH_noncomps, non_comp_weights)) / sum(weights)


        # Calculate the actual_p (observed f)values ########################################
        flip_actual_p_total = flip_with_DMC_count / len_all_seqs
        flip_with_DMC_weights_sum = sum((1 if seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2 else 0) * w for seq, w in zip(flip_seqs, flip_weights))
        flip_actual_p_weighted_total = flip_with_DMC_weights_sum / sum(weights) if flip_seqs else 0
        flip_actual_p_subcategory = flip_with_DMC_count / flip_counts if flip_counts > 0 else 0
        flip_actual_p_weighted_subcategory = sum((1 if seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2 else 0) * w for seq, w in zip(flip_seqs, flip_weights)) / sum(flip_weights) if flip_seqs else 0

        non_flip_actual_p_total = non_flip_with_DMC_count / len_all_seqs
        non_flip_with_DMC_weights_sum = sum((1 if seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2 else 0) * w for seq, w in zip(non_flip_seqs, non_flip_weights))
        non_flip_actual_p_weighted_total = non_flip_with_DMC_weights_sum / sum(weights) if non_flip_seqs else 0
        non_flip_actual_p_subcategory = non_flip_with_DMC_count / non_flip_counts if non_flip_counts > 0 else 0
        non_flip_actual_p_weighted_subcategory = sum((1 if seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2 else 0) * w for seq, w in zip(non_flip_seqs, non_flip_weights)) / sum(non_flip_weights) if non_flip_seqs else 0

        ##############
        gof_actual_p_total = gof_with_DMC_count / len_all_seqs
        gof_with_DMC_weights_sum = sum((1 if seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2 else 0) * w for seq, w in zip(gof_seqs, gof_weights))
        gof_actual_p_weighted_total = gof_with_DMC_weights_sum / sum(weights) if gof_seqs else 0
        gof_actual_p_subcategory = gof_with_DMC_count / gof_counts if gof_counts > 0 else 0
        gof_actual_p_weighted_subcategory = sum((1 if seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2 else 0) * w for seq, w in zip(gof_seqs, gof_weights)) / sum(gof_weights) if gof_seqs else 0

        rescue_actual_p_total = rescue_with_DMC_count / len_all_seqs
        rescue_with_DMC_weights_sum = sum((1 if seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2 else 0) * w for seq, w in zip(rescue_seqs, rescue_weights))
        rescue_actual_p_weighted_total = rescue_with_DMC_weights_sum / sum(weights) if rescue_seqs else 0
        rescue_actual_p_subcategory = rescue_with_DMC_count / rescue_count if rescue_count > 0 else 0
        rescue_actual_p_weighted_subcategory = sum((1 if seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2 else 0) * w for seq, w in zip(rescue_seqs, rescue_weights)) / sum(rescue_weights) if rescue_seqs else 0


        comp_actual_p_total = compensatory_with_DMC_count / len_all_seqs
        comp_with_DMC_weights_sum = sum((1 if seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2 else 0) * w for seq, w in zip(comp_seqs, comp_weights))
        comp_actual_p_weighted_total = comp_with_DMC_weights_sum / sum(weights) if comp_seqs else 0
        comp_actual_p_subcategory = compensatory_with_DMC_count / compensatory_counts if compensatory_counts > 0 else 0
        comp_actual_p_weighted_subcategory = sum((1 if seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2 else 0) * w for seq, w in zip(comp_seqs, comp_weights)) / sum(comp_weights) if comp_seqs else 0

        noncomp_actual_p_total = noncompensatory_with_DMC_count / len_all_seqs
        noncomp_with_DMC_weights_sum = sum((1 if seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2 else 0) * w for seq, w in zip(noncomp_seqs, non_comp_weights))
        noncomp_actual_p_weighted_total = noncomp_with_DMC_weights_sum / sum(weights) if noncomp_seqs else 0
        noncomp_actual_p_subcategory = noncompensatory_with_DMC_count / noncompensatory_counts if noncompensatory_counts > 0 else 0
        noncomp_actual_p_weighted_subcategory = sum((1 if seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2 else 0) * w for seq, w in zip(noncomp_seqs, non_comp_weights)) / sum(non_comp_weights) if noncomp_seqs else 0

        # Append data for each epistasis subset
        #non-weighted
        csv_data.append(['total'])
        csv_data.append([pair, 'Gain_of_function', gof_counts,gof_with_DMC_count, average_p_gof_total, gof_actual_p_total, average_p_gof_subcategory, gof_actual_p_subcategory])
        csv_data.append([pair, 'rescue', rescue_count,rescue_with_DMC_count, average_p_rescue_total, rescue_actual_p_total, average_p_rescue_subcategory, rescue_actual_p_subcategory])
        csv_data.append([pair, 'compensatory', compensatory_counts,compensatory_with_DMC_count, average_p_compensatory_total, comp_actual_p_total, average_p_compensatory_subcategory, comp_actual_p_subcategory])
        csv_data.append([pair, 'non_compensatory', noncompensatory_counts, noncompensatory_with_DMC_count, average_p_noncomp_total, noncomp_actual_p_total, average_p_non_comp_subcategory, noncomp_actual_p_subcategory])
        csv_data.append([])
        # csv_data.append(['total'])
        csv_data.append([pair, 'flipped', flip_counts, flip_with_DMC_count, average_p_flip_total, flip_actual_p_total, average_p_flip_subcategory, flip_actual_p_subcategory])
        csv_data.append([pair, 'non_flipped', non_flip_counts, non_flip_with_DMC_count, average_p_nonflip_total, non_flip_actual_p_total, average_p_nonflip_subcategory, non_flip_actual_p_subcategory])
        csv_data.append([])
        #weighted
        csv_weighted_data.append(['total'])
        csv_weighted_data.append([pair,'Gain_of_function', sum(gof_weights),gof_with_DMC_weights_sum, weighted_gof_prob_total, gof_actual_p_weighted_total, weighted_gof_prob_subcategory, gof_actual_p_weighted_subcategory])
        csv_weighted_data.append([pair,'rescue', sum(rescue_weights), rescue_with_DMC_weights_sum, weighted_rescue_prob_total, rescue_actual_p_weighted_total, weighted_rescue_prob_subcategory, rescue_actual_p_weighted_subcategory])
        csv_weighted_data.append([pair,'compensatory', sum(comp_weights), comp_with_DMC_weights_sum, weighted_comp_prob_total, comp_actual_p_weighted_total, weighted_comp_prob_subcategory, comp_actual_p_weighted_subcategory])
        csv_weighted_data.append([pair,'non_compensatory', sum(non_comp_weights), noncomp_with_DMC_weights_sum, weighted_noncomp_prob_total, noncomp_actual_p_weighted_total, weighted_noncomp_prob_subcategory, noncomp_actual_p_weighted_subcategory])
        csv_weighted_data.append([])
        # csv_weighted_data.append(['total'])
        csv_weighted_data.append([pair,'flipped', sum(flip_weights), flip_with_DMC_weights_sum, weighted_flip_prob_total, flip_actual_p_weighted_total, weighted_flip_prob_subcategory, flip_actual_p_weighted_subcategory])
        csv_weighted_data.append([pair,'non_flipped', sum(non_flip_weights), non_flip_with_DMC_weights_sum, weighted_nonflip_prob_total, non_flip_actual_p_weighted_total, weighted_nonflip_prob_subcategory, weighted_nonflip_prob_subcategory])
        csv_weighted_data.append([])

        with open(output_csv, 'w', newline='') as csvfile:
            csv_writer = csv.writer(csvfile)
            csv_writer.writerow(['mutation_pair', 'epistasis_subset', 'num_seqs', 'num_with_DMC','average_p_total', 'observed_f_total','average_p_subcategory', 'observed_f_subcategory','', 'weights_sum','weights_sum_with_DMC','weighted_average_p_total', 'weighted_observed_f_total', 'weighted_average_p_subcategory', 'weighted_observed_f_subcategory'])
            count = 0 
            for row, weighted_row in zip(csv_data, csv_weighted_data):
                if row == [] or weighted_row == []:
                    csv_writer.writerow([])
                elif row == ['total'] or weighted_row == ['total']:
                    #csv_total_data.append([IN_pair,'Total', total_counts, average_p_all, actual_p_all, weighted_all_prob, actual_p_all_weighted])
                    csv_writer.writerow([f"{csv_total_data[count][0]}", f"{csv_total_data[count][1]}", f"{csv_total_data[count][2]}", f"{csv_total_data[count][3]:}", f"{csv_total_data[count][4]:.4f}", f"{csv_total_data[count][5]:.4f}", f"{csv_total_data[count][6]:.4f}",f"{csv_total_data[count][7]:.4f}", '', f"{csv_total_data[count][8]:.4f}", f"{csv_total_data[count][9]:.4f}", f"{csv_total_data[count][10]:.4f}", f"{csv_total_data[count][11]:.4f}", f"{csv_total_data[count][12]:.4f}", f"{csv_total_data[count][13]:.4f}"])
                    count += 1
                else:
                    csv_writer.writerow([f"{row[0]}", f"{row[1]}", f"{row[2]}", f"{row[3]}", f"{row[4]:.4f}", f"{row[5]:.4f}", f"{row[6]:.4f}", f"{row[7]:.4f}", '', f"{weighted_row[2]:.4f}", f"{weighted_row[3]:.4f}", f"{weighted_row[4]:.4f}", f"{weighted_row[5]:.4f}", f"{weighted_row[6]:.4f}", f"{weighted_row[7]:.4f}"])
            # Append the total data for each mutation pair
    print(f"CSV file {output_csv} created successfully.")

In [8]:
IN_weights_path = 'IN/data/in.weights.txt'
len_IN_all_seqs = len(IN_all_seq)
with open(IN_weights_path, 'r') as f:
    IN_weights = [float(line.strip()) for line in f]

# Ensure the weights list matches the IN_all_seq list
assert len(IN_weights) == len_IN_all_seqs, "Weights and sequences must have the same length."

IN_pairs = [
    'G140S-Q148H',
    'Y143C-S230R',
    'G140A-Q148K',
    'G140S-Q148R',
    'G140S-Q148K',
    'G140A-Q148R',
    'E138K-Q148K',
    'G140A-Q148H',
    'E138K-Q148R',
    'Y143C-S230K',
    'N155H-E170A',
    'E138K-S147G',
    'S147G-Q148R',
    'Y143R-I151V',
    'E138K-Q148H',
    'S147G-L158V',
    'E92Q-K215R',
    'E138A-Q148H',
    'E138K-S230K',
    'E138K-Y194C',
]
output_probs('IN', 1,263,IN_all_seq, IN_consensus_seq, IN_redux, IN_pairs, IN_weights_path, IN_J, 'integrase_all_probabilities_v11.csv')

Processing pair: G140S-Q148H
Processing sequence 1/1220
Processing sequence 2/1220
Processing sequence 3/1220
Processing sequence 4/1220
Processing sequence 5/1220
Processing sequence 6/1220
Processing sequence 7/1220
Processing sequence 8/1220
Processing sequence 9/1220
Processing sequence 10/1220
Processing sequence 11/1220
Processing sequence 12/1220
Processing sequence 13/1220
Processing sequence 14/1220
Processing sequence 15/1220
Processing sequence 16/1220
Processing sequence 17/1220
Processing sequence 18/1220
Processing sequence 19/1220
Processing sequence 20/1220
Processing sequence 21/1220
Processing sequence 22/1220
Processing sequence 23/1220
Processing sequence 24/1220
Processing sequence 25/1220
Processing sequence 26/1220
Processing sequence 27/1220
Processing sequence 28/1220
Processing sequence 29/1220
Processing sequence 30/1220
Processing sequence 31/1220
Processing sequence 32/1220
Processing sequence 33/1220
Processing sequence 34/1220
Processing sequence 35/1220


In [9]:
PR_weights_path = 'PR/data/pr.exper.weights.txt'
len_PR_all_seqs = len(PR_all_seq)
with open(PR_weights_path, 'r') as f:
    PR_weights = [float(line.strip()) for line in f]

# Ensure the weights list matches the IN_all_seq list
assert len(PR_weights) == len_PR_all_seqs, "Weights and sequences must have the same length."

PR_pairs = [
    'D30N-N88D',
    'V32I-I47V',
    'G48V-I54A',
    'D30N-K45Q',
    'I54A-V82A',
    'I54V-V82A',
    'M46I-L76V',
    'I54V-V82T',
    'I54A-V82T',
    'G48V-V82A',
    'I54A-A71I',
    'M46I-N88T',
    'L90M-C95F',
    'V32I-M46I',
    'G48V-V82T',
    'I54V-T91S',
    'M46I-F53Y',
    'M46L-K55R',
    'M46L-V82A',
    'M46I-K55R',
]
output_probs('PR', 1,99,PR_all_seq, PR_consensus_seq, PR_redux, PR_pairs, PR_weights_path, PR_J, 'protease_all_probabilities_v11.csv')

Processing pair: D30N-N88D
Processing sequence 1/5710
Processing sequence 2/5710
Processing sequence 3/5710
Processing sequence 4/5710
Processing sequence 5/5710
Processing sequence 6/5710
Processing sequence 7/5710
Processing sequence 8/5710
Processing sequence 9/5710
Processing sequence 10/5710
Processing sequence 11/5710
Processing sequence 12/5710
Processing sequence 13/5710
Processing sequence 14/5710
Processing sequence 15/5710
Processing sequence 16/5710
Processing sequence 17/5710
Processing sequence 18/5710
Processing sequence 19/5710
Processing sequence 20/5710
Processing sequence 21/5710
Processing sequence 22/5710
Processing sequence 23/5710
Processing sequence 24/5710
Processing sequence 25/5710
Processing sequence 26/5710
Processing sequence 27/5710
Processing sequence 28/5710
Processing sequence 29/5710
Processing sequence 30/5710
Processing sequence 31/5710
Processing sequence 32/5710
Processing sequence 33/5710
Processing sequence 34/5710
Processing sequence 35/5710
Pr

In [10]:
RT_weights_path = 'RT/data/rt.weights.txt'
len_RT_all_seqs = len(RT_all_seq)
with open(RT_weights_path, 'r') as f:
    RT_weights = [float(line.strip()) for line in f]

# Ensure the weights list matches the IN_all_seq list
assert len(RT_weights) == len_RT_all_seqs, "Weights and sequences must have the same length."

RT_pairs = [
    'K101E-G190S',
    'K101E-G190A',
    'K103N-P225H',
    'L100I-K103N',
    'K101P-K103S',
    'Y181C-H221Y',
    'K103S-G190A',
    'K103S-P225H',
    'L100I-K103R',
    'V108I-H221Y',
    'K103S-D192N',
    'L100I-K103S',
    'K101E-E138A',
    'Y181C-G190A',
    'K103S-D177N',
    'V108I-V189I',
    'K101E-E138K',
    'E138A-G190E',
    'K101P-D192N',
    'V108I-L109V',
]

output_probs('NNRTI', 39,226,RT_all_seq, RT_consensus_seq, RT_redux, RT_pairs, RT_weights_path, RT_J, 'reverseTranscriptase_NNRTI_probabilities_v11.csv')

Processing pair: K101E-G190S
Processing sequence 1/19194
Processing sequence 2/19194
Processing sequence 3/19194
Processing sequence 4/19194
Processing sequence 5/19194
Processing sequence 6/19194
Processing sequence 7/19194
Processing sequence 8/19194
Processing sequence 9/19194
Processing sequence 10/19194
Processing sequence 11/19194
Processing sequence 12/19194
Processing sequence 13/19194
Processing sequence 14/19194
Processing sequence 15/19194
Processing sequence 16/19194
Processing sequence 17/19194
Processing sequence 18/19194
Processing sequence 19/19194
Processing sequence 20/19194
Processing sequence 21/19194
Processing sequence 22/19194
Processing sequence 23/19194
Processing sequence 24/19194
Processing sequence 25/19194
Processing sequence 26/19194
Processing sequence 27/19194
Processing sequence 28/19194
Processing sequence 29/19194
Processing sequence 30/19194
Processing sequence 31/19194
Processing sequence 32/19194
Processing sequence 33/19194
Processing sequence 34/

KeyboardInterrupt: 

In [ ]:
RT_weights_path = 'RT/data/rt.weights.txt'
len_RT_all_seqs = len(RT_all_seq)
with open(RT_weights_path, 'r') as f:
    RT_weights = [float(line.strip()) for line in f]

# Ensure the weights list matches the IN_all_seq list
assert len(RT_weights) == len_RT_all_seqs, "Weights and sequences must have the same length."

RT_pairs = [
    'F116Y-Q151M',
    'M41L-T215Y',
    'V75I-I132L',
    'K70R-K219E',
    'D67N-K219Q',
    'K70R-K219Q',
    'L210W-T215Y',
    'F116Y-Q151L',
    'K65R-S68N',
    'V75I-F77L',
    'M41L-T215F',
    'D67N-K219E',
    'L210W-T215S',
    'M41L-T215S',
    'D67N-K70R',
    'L74V-Y115F',
    'L74V-L100I',
    'A62V-V75I',
    'F116Y-Q151R',
    'A62V-V75T',
]
output_probs('PR', 39,226,RT_all_seq, RT_consensus_seq, RT_redux, RT_pairs, RT_weights_path, RT_J, 'reverseTranscriptase_NRTI_probabilities_v11.csv')

Processing pair: F116Y-Q151M
Processing pair: M41L-T215Y
Processing pair: V75I-I132L
Processing pair: K70R-K219E
Processing pair: D67N-K219Q
Processing pair: K70R-K219Q
Processing pair: L210W-T215Y
Processing pair: F116Y-Q151L
Processing pair: K65R-S68N
Processing pair: V75I-F77L
Processing pair: M41L-T215F
Processing pair: D67N-K219E
Processing pair: L210W-T215S
Processing pair: M41L-T215S
Processing pair: D67N-K70R
Processing pair: L74V-Y115F
Processing pair: L74V-L100I
Processing pair: A62V-V75I
Processing pair: F116Y-Q151R
Processing pair: A62V-V75T
CSV file reverseTranscriptase_NRTI_probabilities_v10.csv created successfully.
